### **Simple Linear Regression Modeling**

In [ ]:
# Adding a linear trend line
sns.regplot(x="var1", y="var2", data=DF, ci=None, scatter_kws={'alpha': 0.5})

# Running a linear regression model(var2:"Response Variable", var1:"Explanatory Variable")
from statsmodels.formula.api import ols 
model = ols("var2 ~ var1", data=DF).fit() 
print(model.params)                                    # var2 = 19.99 + 3.41 ∗ var1

# For CATEGORICAL variables ("... + 0")
model0 = ols("var2 ~ CATvar + 0", data=DF).fit()
# In case of a single, categorical variable, coefficients are the means. (same as the calculations using ".groupby()[].mean()")

### **Predictions and model objects**

In [ ]:
# Data on explanatory values to predict (Predicting inside a DataFrame)
explanatory_data = pd.DataFrame({"var1": list_of_values}) 
# for example: {"length_cm": np.arange(20, 41)}
predictions = model.predict(explanatory_data)         # sometimes enough until printing it
prediction_data = explanatory_data.assign(var2=predictions)

# NOTE: Extrapolating means making predictions outside the range of observed data.
IMPOSSIBLE = pd.DataFrame({"length_cm": anything unusal}) # [...,...,...]

import matplotlib.pyplot as plt 
import seaborn as sns 
fig = plt.figure()                                     # Showing predictions
sns.regplot(x="var1",y="var2", ci=None, data=DF,) 
sns.scatterplot(x="var1", y="var2", data=prediction_data, color="red", marker="s") 
plt.show() 
------------------------------------------------------------------------------------------
# Working with model attributes
model.params          # Parameters: Slope and Intercept
model.fittedvalues    # Fitted values: predictions on the original dataset
model.reside          # Residuals: actual response values minus predicted response values
model.summary()
------------------------------------------------------------------------------------------
# Extreme cases are often due to randomness. "Regression to the mean" means extreme cases don't persist over time.
# Due to regression to the mean, it's common that one player or team that does really well one year, doesn't do as well the next. Likewise players or teams that perform very badly one year do better the next year.

fig = plt.figure()
plt.axline(xy1=(0,0), slope=1, linewidth=2, color="green")
sns.regplot(x = "return_2018", y = "return_2019", data = sp500_yearly_returns)
plt.axis("equal") 
plt.show()

# NOTE: Sometimes you can transform explanatory OR response variables into the other forms to showcase the more obviuos regression models and regplots. (For prediction such procedure is needed.)

### **Assessing model fit**

In [ ]:
# Assessing Model Fit:
# 1. Coefficient of determination (R-squared): Measure of how well the linear regression line fits the observed values. (1:perfect, 0:worst)
model.rsquared

# 2. Residual standard error (RSE): "typical" difference between a prediction and an observed response. (same unit as the response variable)
mse = model.mse_resid 
rse = np.sqrt(mse) 
------------------------------------------------------------------------------------------
# Visualizing Model Fit:
# 1. Residuals vs. fitted (residplot): Check whether the model has captured the systematic relationship between X and y. (Randomly scattered around zero → good sign)
sns.residplot(x="var1", y="var2", data=DF, lowess=True)
plt.xlabel("Fitted values") 
plt.ylabel("Residuals")  

# 2. Q-Q plot: asks "Do my residuals look approximately normally distributed?" (Points approximately follow the diagonal → residuals are approximately normal.)
from statsmodels.api import qqplot 
qqplot(data=model.resid, fit=True, line="45")

# 3. Scale-Location Plot: asks "Does the variance of my errors stay roughly constant?" (The vertical spread should remain roughly constant across the X-axis.)
fitted = model.fittedvalues
standardized_resid = model.get_influence().resid_studentized_internal 
sns.regplot(x=fitted, y=np.sqrt(np.abs(standardized_resid)), ci=None, lowess=True) 
plt.xlabel("Fitted values") 
plt.ylabel("Sqrt of abs val of stdized residuals")
------------------------------------------------------------------------------------------
# Leverage: measures how unusual an observation's predictor values (X) are compared with the rest of the dataset. (High leverage = observation is far from the center of the X)
model = ols("var2 ~ var1", data=DF).fit() 
summary_DF = model.get_influence().summary_frame() 
DF["leverage"] = summary_DF["hat_diag"]  # hat_diag: most common influence measures

# Influence: measures how much an individual observation affects the fitted regression model.
DF["cooks_dist"] = summary_DF["cooks_d"] # Cook's distance: most common influence measures
print(DF.sort_values("cooks_dist", ascending = False)) 

### **Simple Logistic Regression Modeling**

In [ ]:
# logistic regression? 
#                     1. Another type of generalized linear model. 
#                     2. Used when the response variable is "logical". 
#                     3. The responses follow logistic (S-shaped) curve. (ZOOM IT OUT!)
from statsmodels.formula.api import logit 
mdl_var2_vs_var1 = logit("var2 ~ var1", data=DF).fit() 

sns.regplot(x="var1", y="var2", data=DF, ci=None, logistic=True)
plt.show()
------------------------------------------------------------------------------------------
# FOUR main ways of expressing the prediction from a logistic regression model
# 1. Probabilities:
prediction_data = explanatory_data.assign(has_churned = model.predict(explanatory_data))

fig = plt.figure()
sns.regplot(x = "var1", y = "var2", data = DF, logistic = True, ci = None)
sns.scatterplot(x = "var1", y = "var2", data = prediction_data, color = "red")
plt.show()

# 2. Most likely outcome: explaining the results to a non-technical audience
prediction_data["most_likely_outcome"] = np.round(prediction_data["has_churned"]) 

# 3. Odds: the probability of smth happening divided by the probability that it doesn't.
prediction_data["odds"] = prediction_data["var2"] / (1 - prediction_data["var2"])
sns.lineplot(x = "var1", y = "odds", data = prediction_data)
plt.axhline(y=1, linestyle="dotted")
plt.show()

# 4. Log odds: does have a linear relationship between predicted response and explanatory variable. One downside to probabilities and odds for logistic regression predictions is that the prediction lines for each are curved.
prediction_data["log_odds"] = np.log(prediction_data["odds"]) 
------------------------------------------------------------------------------------------
# NEEDS COMPLETION